# MCQ Generation SFT — Qwen3-4B with Unsloth (fixed)

Fine-tune Qwen3-4B-Instruct on the MCQ bank.

**Fixes vs. first version:**
- Uses `train_on_responses_only` so loss is computed only on the assistant turn — model learns to emit `<|im_end|>` cleanly and stop on its own
- Inference uses both `<|im_end|>` and `<|endoftext|>` as stop tokens
- Output post-processor extracts the first valid JSON block, dropping any trailing garbage if it leaks through

**Hardware:** RTX 8000 (48 GB VRAM, Turing → fp16 only).


## 1. Install

In [1]:
%%capture
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets
!pip install -q pandas openpyxl scikit-learn


## 2. Load Qwen3-4B base + QLoRA

In [2]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 8192
MODEL_NAME  = "unsloth/Qwen3-4B-Base"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = True,
    full_finetuning = False,
)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/mabdrabou/.pyenv/versions/3.10.13/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[unsloth.import_fixes|WARNING]Unsloth: Detected broken vLLM binary extension; disabling vLLM imports and continuing import.
Please reinstall via `uv pip install unsloth vllm torchvision torchaudio --torch-backend=auto`.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.13.0.
   \\   /|    Quadro RTX 8000. Num GPUs = 1. Max memory: 47.267 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-4b-base-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
GPU: Quadro RTX 8000
VRAM: 50.8 GB
bf16 supported: False


In [3]:
# Attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 32,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
    use_rslora     = False,
    loftq_config   = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")


Unsloth 2026.5.5 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Trainable params: 33,030,144 / 2,464,939,520 (1.34%)


### Attach chat template and verify special tokens

Qwen3 uses ChatML (`<|im_start|>` / `<|im_end|>`). This cell:
1. Attaches the template if not already present
2. **Verifies** that the special tokens have real IDs (not None / `<unk>`)

If the verification fails, EOS won't fire correctly during generation. Re-running
the template attachment usually fixes it.

In [4]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

# Diagnostic — these MUST resolve to integer IDs, not None or unk
im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
im_end_id   = tokenizer.convert_tokens_to_ids("<|im_end|>")
unk_id      = tokenizer.unk_token_id

print(f"<|im_start|> token ID: {im_start_id}")
print(f"<|im_end|>   token ID: {im_end_id}")
print(f"EOS token: {tokenizer.eos_token!r} = id {tokenizer.eos_token_id}")
print(f"PAD token: {tokenizer.pad_token!r} = id {tokenizer.pad_token_id}")

assert im_start_id is not None and im_start_id != unk_id, "<|im_start|> is not a registered special token!"
assert im_end_id   is not None and im_end_id   != unk_id, "<|im_end|> is not a registered special token!"

# Sanity-render the chat template on a tiny example
demo = tokenizer.apply_chat_template(
    [{"role": "user", "content": "ping"}],
    tokenize=False, add_generation_prompt=True,
)
print(f"\nTemplate render demo:\n{demo!r}")


<|im_start|> token ID: 151644
<|im_end|>   token ID: 151645
EOS token: '<|endoftext|>' = id 151643
PAD token: '<|PAD_TOKEN|>' = id 151669

Template render demo:
'<|im_start|>user\nping<|im_end|>\n<|im_start|>assistant\n'


## 3. Load lecture notes (for grounding context)

In [5]:
import json, glob, re

NOTES_GLOB = "/home/mabdrabou/Desktop/GenAI Project/Dataset/Lectures Note/llm_lecture_*_slide_explanations.json"
notes_paths = sorted(glob.glob(NOTES_GLOB))
assert len(notes_paths) == 9, (
    f"Expected 9 lecture JSON files matching {NOTES_GLOB!r}, found {len(notes_paths)}: {notes_paths}"
)

slide_lookup = {}
for path in notes_paths:
    m = re.search(r"lecture_(\d+)", path)
    lec_num = int(m.group(1))
    with open(path, "r", encoding="utf-8") as f:
        slides = json.load(f)
    for slide_idx, slide in enumerate(slides, start=1):
        slide_id = f"L{lec_num}-S{slide_idx}"
        slide_lookup[slide_id] = slide

print(f"Loaded {len(notes_paths)} lecture files | {len(slide_lookup)} slides indexed")
print(f"Sample: L1-S14 -> {slide_lookup['L1-S14']['slide_title']}")


Loaded 9 lecture files | 664 slides indexed
Sample: L1-S14 -> Markov Assumption


## 4. Load MCQ bank

In [6]:
import pandas as pd

MCQ_PATH = "/home/mabdrabou/Desktop/GenAI Project/Dataset/MCQ Exam/llm_mcq_exam_bank.xlsx"
mcq_df = pd.read_excel(MCQ_PATH, sheet_name="MCQ_Bank")

print(f"MCQ bank: {len(mcq_df)} rows")
print(f"\nQuestion type:\n{mcq_df['question_type'].value_counts().to_string()}")
print(f"\nDifficulty:\n{mcq_df['difficulty'].value_counts().to_string()}")
print(f"\nBloom:\n{mcq_df['bloom_level'].value_counts().to_string()}")


MCQ bank: 2111 rows

Question type:
question_type
single_answer    1656
true_false        276
multi_answer      179

Difficulty:
difficulty
Intermediate    887
Beginner        664
Difficult       560

Bloom:
bloom_level
Remember      579
Understand    491
Analyze       442
Apply         291
Evaluate      196
Create        112


## 5. Parsers + SFT formatter

In [7]:
SYSTEM_PROMPT = """You are an exam-question writer for a university course on Generative AI and Large Language Models. Given lecture content and a generation specification, produce exactly one multiple-choice question in valid JSON format.

Rules:
- Use ONLY information present in the lecture content. Do not invent facts.
- Match the requested question_type exactly:
  * single_answer: 4 options labeled A-D, exactly one correct
  * true_false: 2 options (True / False), exactly one correct
  * multi_answer: 4 or more options, multiple correct (correct_answer lists letters)
- Wrong options must target real misconceptions a student might have.
- Include slide citations in evidence_slide_ids using format L{lecture}-S{slide}.
- Output ONLY the JSON object. Do not add any text before or after it."""


def parse_options(options_str):
    if not isinstance(options_str, str):
        return {}
    pattern = re.compile(r"^([A-Z])\.\s*(.+?)(?=\n[A-Z]\.|\Z)",
                          re.MULTILINE | re.DOTALL)
    return {letter: text.strip() for letter, text in pattern.findall(options_str)}


def parse_wrong_explanations(text):
    if not isinstance(text, str):
        return {}
    pattern = re.compile(r"^([A-Z])\.\s*(.+?)(?=\n[A-Z]\.|\Z)",
                          re.MULTILINE | re.DOTALL)
    return {letter: expl.strip() for letter, expl in pattern.findall(text)}


def split_semi_list(s):
    if not isinstance(s, str) or not s.strip():
        return []
    out = []
    for p in re.split(r"[;,]", s):
        head = p.split(":")[0].strip()
        if head:
            out.append(head)
    return out


def build_context(evidence_slide_ids_str):
    sids = split_semi_list(evidence_slide_ids_str)
    if not sids:
        return "(no specific slides cited)"
    parts = []
    for sid in sids:
        if sid in slide_lookup:
            s = slide_lookup[sid]
            parts.append(f"[{sid}: {s['slide_title']}]\n{s['slide_paragraph']}")
    return "\n\n".join(parts) if parts else "(no matching slides found)"


def format_for_sft(row):
    context = build_context(row["evidence_slide_ids"])

    user_msg = (
        f"Lecture content:\n{context}\n\n"
        f"Generation specification:\n"
        f"- Target concept: {row['tested_concepts']}\n"
        f"- Difficulty: {row['difficulty']}\n"
        f"- Bloom level: {row['bloom_level']}\n"
        f"- Question type: {row['question_type']}\n\n"
        f"Generate one MCQ as JSON."
    )

    mcq_output = {
        "question":           row["question"],
        "options":            parse_options(row["options"]),
        "correct_answer":     row["correct_answer"],
        "correct_letter":     row["correct_letter"],
        "difficulty":         row["difficulty"],
        "bloom_level":        row["bloom_level"],
        "question_type":      row["question_type"],
        "tested_concepts":    split_semi_list(row["tested_concepts"]),
        "evidence_slide_ids": split_semi_list(row["evidence_slide_ids"]),
        "why_correct":        row["why_correct"],
        "why_each_wrong_option_is_plausible": parse_wrong_explanations(
            row["why_each_wrong_option_is_plausible"]
        ),
    }

    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": json.dumps(mcq_output, ensure_ascii=False, indent=2)},
        ]
    }


In [8]:
# Sanity-check one parsed row
sample = format_for_sft(mcq_df.iloc[0])
print("=== USER MESSAGE (preview) ===")
print(sample["messages"][1]["content"][:1200])
print("\n=== ASSISTANT MESSAGE (target output, preview) ===")
print(sample["messages"][2]["content"][:1200])


=== USER MESSAGE (preview) ===
Lecture content:
[L4-S12: LLM fine-tuning at a high level - prompt-completion pairs]
This slide refines the fine-tuning dataset format by showing PROMPT and COMPLETION pairs. In instruction tuning, the prompt contains the task instruction and input, while the completion contains the desired model response. This is more natural for LLMs than a simple feature-label table because it matches the way users interact with models at inference time. The diagram shows a pre-trained LLM being fine-tuned on task-specific examples and producing improved performance. The key intuition is that the model learns not only the correct answers, but also how to interpret task instructions, maintain output style, and follow expected response formats.

[L4-S13: Using prompts to fine-tune LLMs with instruction - classification example]
This slide demonstrates that each training pair can include an explicit instruction. The examples use sentiment classification: one prompt asks a

## 6. Build train/val splits

In [9]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
import numpy as np

mcq_df_clean = mcq_df.dropna(subset=["difficulty", "bloom_level",
                                      "question_type", "question", "options"])
strata = mcq_df_clean["difficulty"].astype(str) + "_" + mcq_df_clean["bloom_level"].astype(str)

train_df, val_df = train_test_split(
    mcq_df_clean, test_size=0.15, random_state=42, stratify=strata
)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")


def make_examples(df_subset):
    out = []
    for _, row in df_subset.iterrows():
        try:
            out.append(format_for_sft(row))
        except Exception as e:
            print(f"Skipping row {row.get('question_id', '?')}: {e}")
    return out


train_examples = make_examples(train_df)
val_examples   = make_examples(val_df)
print(f"Built {len(train_examples)} train and {len(val_examples)} val examples")

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )}

train_ds = Dataset.from_list(train_examples).map(to_text, remove_columns=["messages"])
val_ds   = Dataset.from_list(val_examples).map(to_text, remove_columns=["messages"])

# Length check
sample_lens = [len(tokenizer.encode(t)) for t in train_ds["text"][:200]]
print(f"\nToken length (sample of 200): "
      f"mean={np.mean(sample_lens):.0f}  "
      f"p95={int(np.percentile(sample_lens, 95))}  "
      f"max={max(sample_lens)}")

# Show ONE rendered training example end-to-end so we can verify <|im_end|> appears
print(f"\n=== One full training text (tail) ===")
print(train_ds["text"][0][-400:])


Train: 1794 | Val: 317
Built 1794 train and 317 val examples


Map: 100%|██████████| 317/317 [00:00<00:00, 8511.96 examples/s]



Token length (sample of 200): mean=1168  p95=1578  max=1743

=== One full training text (tail) ===
 is C. ~20% lower loss.",
    "D": "Choosing this would show a misconception related to this concept. It is plausible because the option says \"~50% lower loss\", but that claim does not describe the mechanism required by the stem and is not the relationship supported by the cited slides. The tested concept is Scaling laws; model size, so the correct answer is C. ~20% lower loss."
  }
}<|im_end|>



## 7. Train — SFT with response-only loss + early stopping

`train_on_responses_only` masks the loss to compute ONLY over the assistant's
response tokens. This is what makes the model learn to emit `<|im_end|>`
properly at the end of each MCQ (which was the bug in the first run).

In [10]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    eval_dataset       = val_ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    args = SFTConfig(
        output_dir                  = "mcq_sft_qwen3_4b",
        num_train_epochs            = 4,
        per_device_train_batch_size = 4,
        per_device_eval_batch_size  = 4,
        gradient_accumulation_steps = 4,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.03,
        lr_scheduler_type           = "cosine",
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        fp16                        = True,
        bf16                        = False,
        optim                       = "adamw_8bit",
        logging_steps               = 10,
        eval_strategy               = "steps",
        eval_steps                  = 50,
        save_strategy               = "steps",
        save_steps                  = 50,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        report_to                   = "none",
        seed                        = 42,
    ),
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience  = 3,
            early_stopping_threshold = 0.001,
        ),
    ],
)

# ----------------------------------------------------------------------
# CRITICAL FIX: mask loss to only train on assistant tokens.
# Without this, the model tries to predict everything including the
# system prompt and user message, which weakens EOS learning and gives
# rise to the trailing garbage tokens you saw in the first run.
# ----------------------------------------------------------------------
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

stats = trainer.train()
print("\n" + "=" * 70)
print(f"Best eval_loss: {trainer.state.best_metric:.4f}")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Total steps run: {trainer.state.global_step}")


Unsloth: Tokenizing ["text"] (num_proc=36): 100%|██████████| 317/317 [00:03<00:00, 84.37 examples/s] 


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Filter (num_proc=36): 100%|██████████| 317/317 [00:01<00:00, 300.41 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,794 | Num Epochs = 4 | Total steps = 452
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
50,0.351700,0.330598
100,0.315200,0.299914
150,0.272600,0.292229
200,0.237200,0.284773
250,0.220100,0.289769
300,0.207300,0.284912
350,0.175100,0.299983


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



Best eval_loss: 0.2848
Best checkpoint: mcq_sft_qwen3_4b/checkpoint-200
Total steps run: 350


## 8. Inference (with proper stop tokens + JSON extraction)

Two safety nets:

1. **`eos_token_id` is a list** containing both `<|im_end|>` and the default EOS,
   so generation halts on either. This is critical for Qwen3 ChatML.
2. **`extract_first_json` post-processor** — if any garbage still leaks through
   (e.g. from incomplete EOS learning), we extract the first complete `{...}`
   block by brace-counting. The MCQ is always one JSON object.

In [11]:
FastLanguageModel.for_inference(model)

# Stop tokens: end-of-turn marker + default EOS, both as IDs
STOP_TOKEN_IDS = [
    tokenizer.convert_tokens_to_ids("<|im_end|>"),
    tokenizer.eos_token_id,
]
STOP_TOKEN_IDS = [t for t in STOP_TOKEN_IDS if t is not None]
print(f"Generation stop tokens: {STOP_TOKEN_IDS}")


def extract_first_json(text: str):
    """Extract the first complete top-level JSON object from text by brace counting.

    Returns the JSON string (parseable). Drops anything after the matching
    closing brace and anything before the opening brace.
    """
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if escape:
            escape = False
            continue
        if ch == "\\":
            escape = True
            continue
        if ch == '"' and not escape:
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None  # unterminated


def generate_mcq(prompt_messages, max_new_tokens=1024):
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        out = model.generate(
            inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            eos_token_id   = STOP_TOKEN_IDS,
            pad_token_id   = tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    gen_tokens = out[0][inputs.shape[-1]:]
    raw = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    json_str = extract_first_json(raw)
    parsed = None
    if json_str is not None:
        try:
            parsed = json.loads(json_str)
        except json.JSONDecodeError:
            parsed = None

    return {
        "raw":      raw,
        "json_str": json_str,
        "parsed":   parsed,
        "n_tokens": len(gen_tokens),
    }


Generation stop tokens: [151645, 151643]


In [12]:
# Generate on 3 held-out val examples and compare to the originals
import random
random.seed(0)
sample_indices = random.sample(range(len(val_examples)), k=3)

for idx in sample_indices:
    ex = val_examples[idx]
    prompt_msgs = ex["messages"][:2]
    original_assistant = ex["messages"][2]["content"]

    print("\n" + "=" * 90)
    print(f"VAL EXAMPLE {idx}")
    print("=" * 90)

    result = generate_mcq(prompt_msgs, max_new_tokens=1024)

    print(f"\nTokens generated: {result['n_tokens']}")
    print(f"JSON parsed cleanly: {result['parsed'] is not None}")

    if result["parsed"] is not None:
        print("\n--- Generated (cleaned) ---")
        print(json.dumps(result["parsed"], ensure_ascii=False, indent=2)[:1800])
    else:
        print("\n--- Raw (JSON parse failed) ---")
        print(result["raw"][:1800])

    print("\n--- Original (target) ---")
    print(original_assistant[:1800])



VAL EXAMPLE 197

Tokens generated: 1024
JSON parsed cleanly: True

--- Generated (cleaned) ---
{
  "question": "In the \"Using prompts to fine-tune LLMs with instruction\" example, what is the exact instruction given to the model for the review \"I loved this DVD!\"?",
  "options": {
    "A": "Classify this review",
    "B": "Translate this review",
    "C": "Summarize this review",
    "D": "Write a longer review"
  },
  "correct_answer": "A. Classify this review",
  "correct_letter": "A",
  "difficulty": "Difficult",
  "bloom_level": "Remember",
  "question_type": "single_answer",
  "tested_concepts": [
    "Instruction fine-tuning",
    "prompt-completion datasets",
    "supervised adaptation"
  ],
  "evidence_slide_ids": [
    "L4-S12",
    "L4-S13",
    "L4-S2"
  ],
  "why_correct": "A) Classify this review is correct because the cited slides directly support the tested concept(s): Instruction fine-tuning; prompt-completion datasets; supervised adaptation. The evidence (L4-S12: L

## 9. Save

In [13]:
LORA_OUT = "/home/mabdrabou/Desktop/GenAI Project/Fine Tuningmcq_sft_qwen3_4b_lora2"
model.save_pretrained(LORA_OUT)
tokenizer.save_pretrained(LORA_OUT)

import os
total = sum(os.path.getsize(os.path.join(LORA_OUT, f)) for f in os.listdir(LORA_OUT))
print(f"Saved LoRA adapter to ./{LORA_OUT}/  ({total/1e6:.1f} MB)")


Saved LoRA adapter to .//home/mabdrabou/Desktop/GenAI Project/Fine Tuningmcq_sft_qwen3_4b_lora2/  (148.1 MB)


In [14]:
# OPTIONAL: merge LoRA into base and save as fp16 (~8 GB single artifact)
# Uncomment when ready to deploy.

# model.save_pretrained_merged(
#     "mcq_sft_qwen3_4b_merged_fp16",
#     tokenizer,
#     save_method = "merged_16bit",
# )
# print("Merged fp16 model saved.")
